In [ ]:
# Importing packages

import numpy as np
import matplotlib.pyplot as plt
import sys
import os
from xgboost import XGBRegressor

In [ ]:
# Import modules

notebook_dir = os.getcwd()
python_dir = os.path.dirname(notebook_dir)
modules_path = os.path.join(python_dir, 'Modules')

sys.path.append(modules_path)

from validation import walk_forward_validation, plot_walk_forward_results, plot_walk_forward_results2
from testing import final_model_evaluation, plot_final_evaluation, plot_final_evaluation2
from read_data import read_data

In [ ]:
# Read dataset

file_name = "combined_data_cleaned_v3.csv"

(DK1_train, DK1_test, DK2_train, DK2_test, DK1_train_weather, DK1_test_weather, 
DK2_train_weather, DK2_test_weather) = read_data(file_name)

In [ ]:
def make_XGB_predict_fn():
    def predict_fn(train_data, known_data, weather_forecast, predict_horizon):

        X_train = train_data.iloc[:, 1:]  # all columns except target
        y_train = train_data.iloc[:, 0]   # target column

        model = XGBRegressor()
        model.fit(X_train, y_train)

        # Predict autoregressively with frozen exogenous features
        last_row = known_data.iloc[-1:].copy()
        last_row = last_row.iloc[:, 1:]

        predictions = []

        for step in range(predict_horizon):
            next_val = model.predict(last_row)[0]
            predictions.append(next_val)
            
            # Update lagged price columns with predicted value
            if 'Price_lag1' in last_row.columns:
                last_row['Price_lag1'] = next_val
            if 'Price_lag24' in last_row.columns:
                last_row['Price_lag24'] = next_val  # simplification
            # All other features (wind, solar, capacity) stay frozen

            weather_step = len(known_data) + step
            if 'WindSpeed' in last_row.columns:
                last_row['WindSpeed'] = weather_forecast.iloc[weather_step]['WindSpeed']
            if 'Radiation' in last_row.columns:
                last_row['Radiation'] = weather_forecast.iloc[weather_step]['Radiation']

        return np.array(predictions)
    return predict_fn

In [ ]:
xgb_predict = make_XGB_predict_fn()

results_xgb_DK1 = walk_forward_validation(
    data_series = DK1_train,
    weather_forecast_series = DK1_train_weather,
    predict_fn = xgb_predict,
    known_prices_series = None,
    training_window = 17520,
    forecast_horizon = 192,
    known_hours = 24,
    stride = 1,
    expanding = False)

(predictions_DK1, actuals_DK1, fold_rmse_DK1, fold_mae_DK1, fold_mape_DK1,
 daily_rmse_DK1, daily_mae_DK1, daily_mape_DK1) = results_xgb_DK1

In [ ]:
title_DK1 = 'Walk-Forward Validation - XGB: Actual vs. Predicted for DK1'

plot_walk_forward_results(
        predictions_DK1, 
        actuals_DK1, 
        title_DK1)

In [ ]:
plot_walk_forward_results2(
    predictions=predictions_DK1,
    actuals=actuals_DK1,
    title='XGBoost Walk-Forward Validation DK1',
    data_series=DK1_train,
    training_window=17520,
    predict_horizon=168,
    stride=1,
    forecast_horizon=192
)

In [ ]:
xgb_predict = make_XGB_predict_fn()

results_xgb_DK2 = walk_forward_validation(
    data_series = DK2_train,
    weather_forecast_series = DK2_train_weather,
    predict_fn = xgb_predict,
    known_prices_series = None,
    training_window = 17520,
    forecast_horizon = 192,
    known_hours = 24,
    stride = 1,
    expanding = False)

(predictions_DK2, actuals_DK2, fold_rmse_DK2, fold_mae_DK2, fold_mape_DK2,
 daily_rmse_DK2, daily_mae_DK2, daily_mape_DK2) = results_xgb_DK2

In [ ]:
title_DK2 = 'Walk-Forward Validation - XGB: Actual vs. Predicted for DK2'

plot_walk_forward_results(
        predictions_DK2, 
        actuals_DK2, 
        title_DK2)

In [ ]:
plot_walk_forward_results2(
    predictions=predictions_DK2,
    actuals=actuals_DK2,
    title='XGBoost Walk-Forward Validation DK2',
    data_series=DK2_train,
    training_window=17520,
    predict_horizon=168,
    stride=1,
    forecast_horizon=192
)

In [ ]:
DK1_X_train_full = DK1_train.iloc[:, 1:]  # all columns except price
DK1_y_train_full = DK1_train.iloc[:, 0]   # price column

# Train once on full training set
DK1_trained_xgb = XGBRegressor()  # or XGBRegressor() if no tuning
DK1_trained_xgb.fit(DK1_X_train_full, DK1_y_train_full)

# Pass to final evaluation
DK1_test_results = final_model_evaluation(
    trained_model=DK1_trained_xgb,
    train_data=DK1_train,
    test_data=DK1_test,
    weather_forecast_series=DK1_test_weather,
    inference_fn=predict_with_shallow_model,
    known_hours=24,
    forecast_horizon=192,
)

(DK1_test_predictions, DK1_test_actuals,
 DK1_test_fold_rmse, DK1_test_fold_mae, DK1_test_fold_smape,
 DK1_test_daily_rmse, DK1_test_daily_mae, DK1_test_daily_smape) = DK1_test_results

In [ ]:
plot_final_evaluation(
    predictions=DK1_test_predictions,
    actuals=DK1_test_actuals,
    title='XGBoost Final Evaluation DK1',
    predict_horizon=168,
)

In [ ]:
plot_final_evaluation2(
    predictions=DK1_test_predictions,
    actuals=DK1_test_actuals,
    title='XGBoost Final Evaluation DK1',
    train_data=DK1_train,
    test_data=DK1_test,
    forecast_horizon=192,
    predict_horizon=168,
)

In [ ]:
DK2_X_train_full = DK2_train.iloc[:, 1:]  # all columns except price
DK2_y_train_full = DK2_train.iloc[:, 0]   # price column

# Train once on full training set
DK2_trained_xgb = XGBRegressor()  # or XGBRegressor() if no tuning
DK2_trained_xgb.fit(DK2_X_train_full, DK2_y_train_full)

# Pass to final evaluation
DK2_test_results = final_model_evaluation(
    trained_model=DK2_trained_xgb,
    train_data=DK2_train,
    test_data=DK2_test,
    weather_forecast_series=DK2_test_weather,
    inference_fn=predict_with_shallow_model,
    known_hours=24,
    forecast_horizon=192,
)

(DK2_test_predictions, DK2_test_actuals,
 DK2_test_fold_rmse, DK2_test_fold_mae, DK2_test_fold_smape,
 DK2_test_daily_rmse, DK2_test_daily_mae, DK2_test_daily_smape) = DK2_test_results

In [ ]:
plot_final_evaluation(
    predictions=DK2_test_predictions,
    actuals=DK2_test_actuals,
    title='XGBoost Final Evaluation DK2',
    predict_horizon=168,
)

In [ ]:
plot_final_evaluation2(
    predictions=DK2_test_predictions,
    actuals=DK2_test_actuals,
    title='XGBoost Final Evaluation DK2',
    train_data=DK2_train,
    test_data=DK2_test,
    forecast_horizon=192,
    predict_horizon=168,
)